# 03 - Feature Engineering and Pre-Processing

##### 03-00 General Model Plan

Feature Engineering

- Create `duration_days` from the cleaned date columns
- Remove invalid and outlier duration rows
- Create the target variable from duration ranges
- Save the final cleaned modeling dataset

Pre-Processing

- Split dataset into training and testing subsets
- Split pre-processing steps into 3 groups (Text, Categories, and Words/Lengths)
- Use TF-IDF vectorization to convert text values (summaries and descriptions) to numeric matrices
- Use One Hot Encoder to encode catgories into numeric values with no relations, e.g, Bug and Improvement are completely unrelated
- Use Standard Scaler to standardizes the numeric values

Model training

- Use Logistic Regression


## 03-01 Importing CSV file and defining the target

Before we start training the model using the cleaned dataset, we need to create the target value from the cleaned date columns.

In [12]:
from pathlib import Path
import pandas as pd

jira_df = pd.read_csv("../data/processed/jira_issues_cleaned.csv", index_col=0)

task_df = jira_df.copy()

for col in ["created", "resolutiondate"]:
    task_df[col] = pd.to_datetime(task_df[col])

task_df.head()

,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,49,9,176,28,2005-01-01 07:47:50,2005-01-01 07:50:46
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,2004-12-25 22:50:30,2004-12-30 05:30:36
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,2005-01-01 13:52:46,2005-01-02 15:21:00
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,2004-12-27 01:34:17,2005-01-03 10:21:28
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,2004-12-28 18:50:52,2005-01-03 11:29:27


## 03-02 Create duration days derived variable

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [13]:
task_df["duration_days"] = (task_df["resolutiondate"] - task_df["created"]).dt.total_seconds() / (60 * 60 * 24)

task_df["duration_days"].describe()

count    938519.000000
mean        201.985744
std         517.790205
min           0.000000
25%           1.042581
50%          11.802130
75%         110.378437
max        8001.517257
Name: duration_days, dtype: float64

The above statistics show target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

#### Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected.

In [14]:
display(task_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

durations_over_120_days = (task_df["duration_days"] > 120).sum()
print(f"Rows with duration_days > 120: {durations_over_120_days:,}")

count    938519.000000
mean        201.985744
std         517.790205
min           0.000000
50%          11.802130
75%         110.378437
90%         617.085472
95%        1184.935810
99%        2689.762594
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 120: 226,804


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [15]:
task_df = task_df[task_df["duration_days"] <= 120].copy()

display(task_df.shape)
task_df.describe()

(711715, 11)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,711715.000000,711715.000000,7.117150e+05,711715.000000,711715,711715,711715.000000
mean,57.160927,7.904872,9.168527e+02,80.222031,2015-10-26 06:37:55.393619,2015-11-12 02:58:40.843381,16.847748
min,0.000000,0.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-05-10 18:54:17,0.000000
25%,40.000000,5.000000,1.180000e+02,17.000000,2012-06-18 10:05:35.500000,2012-07-03 10:39:24.500000,0.533773
50%,54.000000,7.000000,2.900000e+02,40.000000,2016-02-03 01:36:51,2016-02-18 21:03:07,4.267951
75%,70.000000,10.000000,6.840000e+02,85.000000,2019-08-15 20:20:50.500000,2019-09-03 14:01:59,20.843084
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 14:09:26,2025-02-26 18:17:12,119.999815
std,24.032472,3.662664,1.180217e+04,603.544202,NaN,NaN,26.389295


Tasks resolved in under 2 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [16]:
task_df = task_df[task_df["duration_days"] >= (2/24)].copy()

## 03-03 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Short:** same day and up to 3 days
- **Standard:** greater than 3 days and up to 3 weeks
- **Long-term:** greater than 3 weeks


In [17]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 21:
        return "Standard"
    return "Long-running"

duration_order = ["Short", "Standard", "Long-running"]

task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_counts = task_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (task_df["duration_category"].value_counts(normalize=True).reindex(duration_order).mul(100).round(2))

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(task_df.shape)
display(task_df.describe())

duration_class_summary

(605812, 12)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,605812.000000,605812.000000,6.058120e+05,605812.000000,605812,605812,605812.000000
mean,57.430847,7.927395,9.688450e+02,84.576019,2015-12-15 09:02:33.346087,2016-01-04 03:58:47.935428,19.789058
min,1.000000,1.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-06-03 18:35:07,0.083333
25%,41.000000,5.000000,1.300000e+02,18.000000,2012-08-15 14:35:24.750000,2012-09-05 16:35:20.500000,1.389404
50%,54.000000,7.000000,3.120000e+02,43.000000,2016-04-20 14:31:19,2016-05-10 00:27:03,6.829572
75%,70.000000,10.000000,7.300000e+02,90.000000,2019-10-30 15:52:04.250000,2019-11-18 18:05:13.750000,26.704925
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 12:43:08,2025-02-26 18:17:12,119.999815
std,24.029017,3.666178,1.255338e+04,636.904097,NaN,NaN,27.567958


,count,percent
duration_category,,
Short,213505,35.24
Standard,215847,35.63
Long-running,176460,29.13


### 03-03.1 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [18]:
sample_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "duration_days",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = task_df.loc[
        task_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Short


,summary,description,priority_name,issuetype_name,duration_days,duration_category
1120620,Native deserializer segfaults on incorrect lis...,There is a pretty major bug in the native Ruby...,Major,Bug,2.744664,Short
912810,Add external endpoint value,Add value (external endpoint) in Selected endp...,Minor,Task,2.043449,Short
125791,Should DefaultPageService#updatePageRenderServ...,TODO added by Matt: //TODO: If there is a reas...,Minor,Technical task,0.139213,Short
663472,Upgrading Hadoop to 2.7.4 to fix java.version ...,When I ran spark-shell on JDK11+28(2018-09-25)...,Major,Sub-task,1.055278,Short
912855,[Configuration]: Discard changes does not work...,*Preconditions:* 1. External endpoint is conne...,Minor,Bug,2.055220,Short


Standard


,summary,description,priority_name,issuetype_name,duration_days,duration_category
267635,The management schema no longer exposes a way ...,The console needs a way to associate links wit...,Major,Bug,8.013634,Standard
634095,Upgrade plexus-archiver to 3.6.0,NaN,Critical,Dependency upgrade,12.043715,Standard
764678,Menu select operation does not get recorded wh...,Steps to reproduce: 1.Run the attached app 2.R...,Major,Bug,20.248009,Standard
120286,Moved logic from commons-management to camel-core,NaN,Major,Sub-task,12.314340,Standard
817760,IndexReader.isCurrent fails when using two Ind...,there is a problem in IndexReader.isCurrent() ...,Minor,Bug,10.048021,Standard


Long-running


,summary,description,priority_name,issuetype_name,duration_days,duration_category
458358,Provide guide for enabling Jel.Camel,[Jel.Camel|https://github.com/giacomolm/jel.ca...,Major,Improvement,41.777928,Long-running
1055588,Upgrade zookeeper to 3.5.5,Upgrade zookeeper dependencies to 3.5.5 and test.,Major,Improvement,29.166493,Long-running
519303,FileSystemInputPartition doesn't throw file co...,"In {{FileSystemInputPartition}}, we check whet...",Major,Improvement,71.081389,Long-running
317061,"""module not found"" when Java 9 module-info pre...",Using my project at https://github.com/meterwa...,Major,Bug,28.141458,Long-running
819874,TestIndexWriterExceptions fails on windows (2),Note: this is a different problem than LUCENE-...,Major,Bug,78.178044,Long-running


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

## 03-04 Save final cleaned dataset

Save the final cleaned dataset after feature engineering.

In [19]:
final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "duration_days",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()
final_cleaned_df_sample = final_cleaned_df.sample(n=100000, random_state=42)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_cleaned_path = output_dir / "final_cleaned.csv"
final_cleaned_df.to_csv(final_cleaned_path, index=False)

final_cleaned_df_sample_path = output_dir / "final_cleaned_sample.csv"
final_cleaned_df_sample.to_csv(final_cleaned_df_sample_path, index=False)

print(f"Final cleaned modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df.shape[1]:,}")
print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved final cleaned sample CSV file to: {final_cleaned_df_sample_path}")

final_cleaned_df.head()

Final cleaned modeling rows: 605,812
Final cleaned modeling columns: 10
Saved final cleaned CSV file to: ..\data\processed\final_cleaned.csv
Saved final cleaned sample CSV file to: ..\data\processed\final_cleaned_sample.csv


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,duration_days,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,4.277847,Standard
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,1.061273,Short
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,7.366100,Standard
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,5.693461,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,32.141562,Long-running


In [9]:
print(f"Final cleaned modeling rows: {final_cleaned_df_sample.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df_sample.shape[1]:,}")

Final cleaned modeling rows: 100,000
Final cleaned modeling columns: 10
